In [23]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv('tuning-results.csv')
df.head()

,data_name,lambda,prerank,pce,energy
0,households,0.00,marginal,0.099625,0.930987
1,households,0.01,marginal,0.098791,0.935000
2,households,0.10,marginal,0.086697,0.932241
3,households,1.00,marginal,0.070050,0.952704
4,households,5.00,marginal,0.031441,0.940808


In [7]:
best_lambdas = {}
for dataset in df['data_name'].unique():
    for prerank in df.prerank.unique():
        subdf = df[(df['data_name'] == dataset) & (df['prerank'] == prerank)]
        if 0.0 not in subdf['lambda'].values:
            continue
        baseline_energy = subdf[subdf['lambda'] == 0.0]['energy'].values[0]
        valid = subdf[subdf['energy'] <= 1.1 * baseline_energy]
        best = valid.sort_values('pce').iloc[0] if not valid.empty else subdf.sort_values('pce').iloc[0]
        best_lambdas[(dataset, prerank)] = best['lambda'] 
best_lambdas

{('households', 'marginal'): np.float64(10.0),
 ('households', 'mean'): np.float64(10.0),
 ('households', 'variance'): np.float64(10.0),
 ('households', 'dependency'): np.float64(5.0),
 ('households', 'pca'): np.float64(10.0),
 ('households', 'density'): np.float64(10.0),
 ('households', 'cdf'): np.float64(5.0),
 ('air', 'marginal'): np.float64(10.0),
 ('air', 'mean'): np.float64(10.0),
 ('air', 'variance'): np.float64(10.0),
 ('air', 'dependency'): np.float64(10.0),
 ('air', 'pca'): np.float64(10.0),
 ('air', 'density'): np.float64(10.0),
 ('air', 'cdf'): np.float64(5.0),
 ('births1', 'marginal'): np.float64(10.0),
 ('births1', 'mean'): np.float64(10.0),
 ('births1', 'variance'): np.float64(10.0),
 ('births1', 'pca'): np.float64(10.0),
 ('births1', 'density'): np.float64(5.0),
 ('births1', 'cdf'): np.float64(10.0),
 ('births2', 'marginal'): np.float64(10.0),
 ('births2', 'mean'): np.float64(10.0),
 ('births2', 'variance'): np.float64(5.0),
 ('births2', 'dependency'): np.float64(5.0),


In [24]:
df = pd.read_csv('metrics-after-reg.csv')
df.head()

,data_name,seed,prerank,pce,nll,energy,mse
0,households,0,marginal,0.028741,3.295859,0.951055,0.663382
1,households,42,marginal,0.031749,3.270183,0.882693,0.554523
2,households,866,marginal,0.029671,3.396923,0.965792,0.652964
3,households,12,marginal,0.029982,3.423873,0.960883,0.667672
4,households,4,marginal,0.027437,3.255520,0.935410,0.623558


In [25]:
df.shape

(175, 7)

In [26]:
agg_df = (
     df.groupby(["data_name", "prerank"])
      .agg(pce_mean=("pce", "mean"),
           pce_se=("pce", lambda x: x.std() / (len(x) ** 0.5)),
           nll_mean=("nll", "mean"),
           nll_se=("nll", lambda x: x.std() / (len(x) ** 0.5)),
           energy_mean=("energy", "mean"),
           energy_se=("energy", lambda x: x.std() / (len(x) ** 0.5)),
           mse_mean=("mse", "mean"),
           mse_se=("mse", lambda x: x.std() / (len(x) ** 0.5)))
          ).reset_index()

In [27]:
agg_df[agg_df['data_name']=='households']

,data_name,prerank,pce_mean,pce_se,nll_mean,nll_se,energy_mean,energy_se,mse_mean,mse_se
14,households,cdf,0.030640,0.000899,3.354427,0.049650,0.952433,0.017981,0.658789,0.030203
15,households,density,0.038680,0.002938,3.292781,0.041308,0.932650,0.016602,0.628786,0.025634
16,households,dependency,0.020640,0.002509,3.228757,0.050194,0.921436,0.013940,0.619659,0.021915
17,households,marginal,0.029516,0.000712,3.328471,0.034329,0.939167,0.015042,0.632420,0.020940
18,households,mean,0.023898,0.001960,3.395145,0.033335,0.944452,0.010778,0.628032,0.021934
19,households,pca,0.025954,0.000801,3.330700,0.066509,0.937192,0.015956,0.621929,0.023278
20,households,variance,0.021100,0.002394,3.335899,0.034272,0.936779,0.012612,0.634394,0.021343


In [30]:
dataset = "households"
subdf = agg_df[agg_df["data_name"] == dataset]
for _, row in subdf.iterrows():
    print(f"""{row['prerank']}: {row['nll_mean']:.3f} ({row['nll_se']:.3f}) & {row['energy_mean']:.3f} ({row['energy_se']:.3f}) & {row['mse_mean']:.3f} ({row['mse_se']:.3f})""")

cdf: 3.354 (0.050) & 0.952 (0.018) & 0.659 (0.030)
density: 3.293 (0.041) & 0.933 (0.017) & 0.629 (0.026)
dependency: 3.229 (0.050) & 0.921 (0.014) & 0.620 (0.022)
marginal: 3.328 (0.034) & 0.939 (0.015) & 0.632 (0.021)
mean: 3.395 (0.033) & 0.944 (0.011) & 0.628 (0.022)
pca: 3.331 (0.067) & 0.937 (0.016) & 0.622 (0.023)
variance: 3.336 (0.034) & 0.937 (0.013) & 0.634 (0.021)


In [50]:
order = ["marginal", "mean", "variance", "dependency", "pca", "density", "cdf"]
subset = agg_df[agg_df["data_name"] == "scm20d"]
subset = subset.set_index("prerank").loc[order]
result = " & ".join(f"{row['mse_mean']:.3f} ({row['mse_se']:.3f})" for _, row in subset.iterrows())
print(result)


0.737 (0.015) & 0.759 (0.017) & 0.834 (0.011) & 0.833 (0.015) & 0.790 (0.022) & 0.851 (0.008) & 0.822 (0.023)
